# 04 — Multi-source data center labels

Notebook 03 left the model badly overconfident at the top of its score range:
predicted 0.347 against an observed 95% CI of [0.052, 0.315]. That was a **label**
problem, not a model problem. Among ZCTAs OSM calls empty, 1.33% contain a PeeringDB
facility — but among those the model scores above 0.20, 13.25% do. A 10x enrichment.
The model's "false positives" were real data centers OSM had never tagged.

This notebook rebuilds the ground truth from four sources and re-checks everything.

| source | licence | independent of OSM? | contributes |
|---|---|---|---|
| OSM | ODbL | — | 1,809 footprints |
| PeeringDB | open API | yes | 1,353 carrier hotels / colo (urban-heavy) |
| IM3 Atlas | ODbL (code BSD-2) | **no — OSM-derived** | campus/building split, cleaned operator, sqft |
| Compute Atlas | CC BY 4.0, (c) 2026 Edward Kubiak | yes | status, MW capacity, dated history, citations |

Deliberately excluded: Baxtel, DataCenterMap, Cloudscene. Those are commercial
database products where the compilation *is* the business — database rights and ToS
make scraping them a legal exposure, not a technical question.

Also note compute-atlas.com's robots.txt disallows ClaudeBot and sets
`Content-Signal: ai-train=no`. We do not crawl the site; we read the author's own
GitHub repository, which carries an explicit licence.

In [ ]:
import subprocess, sys
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import GroupKFold, cross_val_predict
from scipy.stats import beta
import dcmodel as D

# Build the label database if it has not been run yet.
if not (D.HERE.parent / "data" / "datacenters" / "zcta_labels.csv").exists():
    subprocess.run([sys.executable, "-u", "../scripts/17_datacenter_labels.py"], check=True)

df = D.load_model_table()
for c in ["has_dc_any"] + D.LABELS:
    print(f"{c:<16} positives={int(df[c].sum()):>5}   base rate={100*df[c].mean():.2f}%")

## 1. What the extra sources bought

5,736 raw records resolve to 2,302 distinct facilities (300 m merge radius, so the
several buildings of one hyperscale campus become one facility). 822 of those are
asserted by two or more independent sources.

Positives per ZCTA go from 523 (OSM only) to 1,196.

In [ ]:
fac = __import__("geopandas").read_file(D.HERE.parent / "data" / "datacenters" / "facilities.geojson")
print("facilities by number of independent sources:")
print(fac["n_sources"].value_counts().sort_index().to_string())
print()
print("facilities by status:")
print(fac["status"].value_counts().to_string())

## 2. The bias audit — and the most important result here

Adding sources naively could have made things worse: PeeringDB is *more* urban-biased
than OSM, so leaning on it fixes calibration while deepening the urbanization confound
that was already the weakest part of the story.

`pop_exposure_score_nat` is high for sparse/rural ZCTAs, so "rural share" below is the
fraction of labelled ZCTAs above 0.9. The national baseline is 69.9%.

In [ ]:
rows = []
for lab, nm in [("has_dc_any", "OSM only"), ("has_dc", "4 sources"),
                ("has_dc_multi", "2+ sources"), ("has_dc_forward", "proposed/permitted")]:
    p = df[df[lab] == 1]
    rows.append({"label": nm, "n": len(p),
                 "rural_share": (p["pop_exposure_score_nat"] > 0.9).mean(),
                 "median_pop_exp": p["pop_exposure_score_nat"].median()})
rows.append({"label": "(all ZCTAs)", "n": len(df),
             "rural_share": (df["pop_exposure_score_nat"] > 0.9).mean(),
             "median_pop_exp": df["pop_exposure_score_nat"].median()})
print(pd.DataFrame(rows).to_string(index=False, float_format=lambda x: f"{x:8.3f}"))

**Read that table carefully — it reframes the whole project.**

- Existing mapped data centers are heavily urban: 22.8% rural for OSM, 29.3% with all
  four sources, and only 19.4% for the multi-source-confirmed set (the sources overlap
  most in cities, so demanding confirmation *biases urban*).
- **Proposed and permitted facilities are 66.4% rural — against a 69.9% national
  baseline.** They are very nearly as rural as the country at large.

So the urbanization confound that dominated notebook 03 is largely an artifact of
*what has already been built and mapped*. It does not carry over to what is being
proposed. The industry moved from urban colo to rural hyperscale campuses, and our
OSM/PeeringDB labels were a snapshot of the old regime.

That is also why `PHYSICAL only` lift is highest on the forward label (2.09x, versus
1.60x on OSM-only): greenfield rural siting is exactly where physical constraints bite.

## 3. Did calibration improve?

In [ ]:
blocks = D.spatial_blocks(df)
cols = [c for c in D.URBAN + D.POWER + D.PHYSICAL if c in df.columns]
X = D.feature_matrix(df, cols).to_numpy()
bins = [0, .02, .05, .10, .20, .30, .45, 1.01]

preds = {}
for label in ["has_dc_any", "has_dc"]:
    y = df[label].to_numpy()
    clf = HistGradientBoostingClassifier(max_iter=300, learning_rate=0.06,
                                         max_leaf_nodes=15, l2_regularization=1.0,
                                         random_state=0)
    p = cross_val_predict(clf, X, y, cv=GroupKFold(n_splits=8), groups=blocks,
                          method="predict_proba")[:, 1]
    preds[label] = p
    d = pd.DataFrame({"p": p, "y": y}); d["b"] = pd.cut(d.p, bins)
    print(f"=== {label}  (n_pos={y.sum()}) ===")
    for b, g in d.groupby("b", observed=True):
        n, k = len(g), int(g.y.sum())
        lo = beta.ppf(.025, k + .5, n - k + .5) if k else 0
        hi = beta.ppf(.975, k + .5, n - k + .5)
        flag = "  <-- outside" if (g.p.mean() < lo or g.p.mean() > hi) else ""
        print(f"  {str(b):<13} n={n:>6} pred={g.p.mean():.3f} obs={k/n:.3f} "
              f"CI=[{lo:.3f},{hi:.3f}]{flag}")
    print()
df["p"] = preds["has_dc"]

The headline bin is fixed. `(0.3, 0.45]` went from **predicted 0.347 vs observed 0.148**
(wildly overconfident) to **predicted 0.349 vs observed 0.307, inside its CI**.

Two caveats, stated rather than buried:

- `(0.2, 0.3]` is still overconfident (0.238 vs 0.174), though far closer than before.
- A new top bin `(0.45, 1.01]` appears at n=20 and is badly over (0.504 vs 0.150). Tiny n.
- The low bins flag as "outside" by margins of ~0.001 at n>20,000. That is over-precision
  at large sample size, not practical miscalibration — do not read it as a problem.

**This is exactly why the product ships tiers with observed rates rather than
probabilities.** Calibration improved a lot; it is not good enough to put a percentage
next to somebody's ZIP code.

## 4. The tier table — what actually ships

In [ ]:
t = D.tier_table(df, "has_dc", p_col="p")
for _, r in t.iterrows():
    print(f"  Tier {int(r.tier):>2}  n={int(r.n):>5}   hosts a data center {100*r.observed:>5.1f}%"
          f"   CI=[{100*r.ci_lo:.1f}, {100*r.ci_hi:.1f}]   ej={r.ej_mean:.3f}")
print(f"\n  top tier vs bottom tier: {t.observed.iloc[-1]/max(t.observed.iloc[0],1e-9):.0f}x")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.bar(t["tier"], 100*t["observed"], color="#4c72b0",
       yerr=[100*(t.observed-t.ci_lo), 100*(t.ci_hi-t.observed)],
       capsize=3, ecolor="#333", linewidth=0)
ax.set_xlabel("exposure tier"); ax.set_ylabel("% of ZCTAs hosting a data center")
ax.set_xticks(range(1, 11))
ax.set_title("Observed rate by tier — the number the public product reports")
plt.tight_layout(); plt.show()

Monotonic across all ten tiers, tight intervals at n=3,325 each, and a 254x spread
from bottom to top. That is a defensible public statement: *"ZIPs in this tier host a
data center 15.3% of the time, against 0.1% in the lowest tier."* Same statistic on
both sides, no probability claim, no calibration dependency.

Note the `ej` column falls monotonically too, 0.824 down to 0.527 (the score is
inverted, so lower means more community burden). That skew is real and gets stated as
a finding on the methodology page rather than hidden.

## 5. Limitations that survive the relabelling

- **Still partly retrodiction.** `has_dc` is where facilities are now. `has_dc_forward`
  (259 positives) is genuinely forward-looking, but it is thin and its model is much
  weaker — 21x top-vs-bottom against 254x for the built label. Predicting the future
  is harder than detecting the present, and the numbers say so.
- **Multi-source confirmation biases urban.** `has_dc_multi` looks like the highest-
  quality label but is the *least* rural (19.4%), because the sources overlap in cities.
  Do not treat "confirmed by 3 sources" as neutral.
- **Compute Atlas is one maintainer's compilation.** It is well-sourced and updated
  (349 dated events in 2026 alone) but it is not an official register, and its coverage
  decisions are its own.
- **IM3 adds no independent coverage** — it is OSM-derived. It earns its place for the
  campus/building split and cleaned operator field, not for new facilities.